# Implement Top-p (Nucleus) Sampling

**Difficulty**: 🟡 Medium

### Problem Statement

Top-p (nucleus) sampling is a decoding strategy used in large language models that dynamically selects the smallest set of tokens whose cumulative probability exceeds a threshold `p`. Unlike top-k sampling which uses a fixed number of tokens, top-p adapts the candidate pool size based on the probability distribution — using fewer tokens when the model is confident and more when it is uncertain.

Your goal is to implement nucleus sampling from scratch: apply temperature scaling, convert logits to probabilities, sort them in descending order, compute the cumulative sum, mask out tokens below the nucleus threshold, renormalize, and sample.

---

### Requirements

1. **Temperature Scaling**
   - Divide logits by the temperature parameter before computing probabilities.

2. **Sort and Cumulative Sum**
   - Convert scaled logits to probabilities via softmax.
   - Sort probabilities in descending order.
   - Compute the cumulative sum of sorted probabilities.

3. **Nucleus Masking**
   - Create a mask for tokens whose cumulative probability exceeds the threshold `p`.
   - Zero out the masked probabilities.
   - Renormalize the remaining probabilities so they sum to 1.

4. **Sampling**
   - Sample a token index from the renormalized distribution using `torch.multinomial`.
   - Map the sampled index back to the original vocabulary position.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ The output should be a single token index (integer).
- ✅ The sampled token must come from within the nucleus (top-p probability mass).
- ✅ Must handle edge cases where temperature is very low.

---

**Companies**: Anthropic, OpenAI, DeepMind, Perplexity, Cohere

---

<details>
  <summary>💡 Hint</summary>

  - Use `torch.sort` with `descending=True` to sort probabilities. It returns both sorted values and the original indices.
  - Use `torch.cumsum` on the sorted probabilities to get the cumulative distribution.
  - Create a boolean mask where `cumsum - sorted_probs >= p` to identify tokens outside the nucleus. Subtracting the current probability ensures the first token that crosses the threshold is kept.
  - After masking, divide by the sum to renormalize. Then use `torch.multinomial` to sample.
  - Use the saved sort indices to map the sampled position back to the original vocabulary index.

</details>

---

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# Data generation / setup
torch.manual_seed(42)
vocab_size = 50257

# Create fake logits as if from an LLM's final layer
logits = torch.randn(1, vocab_size)
print(f"Logits shape: {logits.shape}")
print(f"Logits range: [{logits.min().item():.4f}, {logits.max().item():.4f}]")

In [ ]:
def top_p_sample(logits, p=0.9, temperature=1.0):
    """
    Perform top-p (nucleus) sampling on logits.
    
    Args:
        logits (Tensor): Raw logits of shape (1, vocab_size)
        p (float): Cumulative probability threshold (0.0 to 1.0)
        temperature (float): Temperature for scaling logits
        
    Returns:
        int: Sampled token index
    """
    # Step 1: Apply temperature scaling
    # TODO: Divide logits by temperature
    ...
    
    # Step 2: Convert to probabilities using softmax
    # TODO: Apply softmax to get probabilities
    ...
    
    # Step 3: Sort probabilities in descending order
    # TODO: Use torch.sort to get sorted probs and original indices
    ...
    
    # Step 4: Compute cumulative sum
    # TODO: Use torch.cumsum on sorted probabilities
    ...
    
    # Step 5: Create nucleus mask and zero out tokens outside the nucleus
    # TODO: Mask where cumsum - current_prob >= p, then zero those out
    ...
    
    # Step 6: Renormalize the remaining probabilities
    # TODO: Divide by sum so probabilities sum to 1
    ...
    
    # Step 7: Sample from the renormalized distribution
    # TODO: Use torch.multinomial to sample, then map back to original index
    ...

In [ ]:
# Validation
print("=== Top-p Sampling Validation ===")
print()

# Test 1: Output should be a valid token index
token = top_p_sample(logits, p=0.9, temperature=1.0)
assert isinstance(token, int) or (isinstance(token, torch.Tensor) and token.ndim == 0), \
    f"Output should be a scalar token index, got {type(token)}"
assert 0 <= token < vocab_size, f"Token {token} out of range [0, {vocab_size})"
print(f"Test 1 PASSED: Sampled token {token} is a valid index")

# Test 2: Sampled tokens should only come from the nucleus
probs = F.softmax(logits, dim=-1)
sorted_probs, sorted_indices = torch.sort(probs, descending=True)
cumsum = torch.cumsum(sorted_probs, dim=-1)
# Find nucleus size (how many tokens are in the top-p set)
nucleus_mask = (cumsum - sorted_probs) < 0.9
nucleus_size = nucleus_mask.sum().item()
nucleus_tokens = set(sorted_indices[0, :nucleus_size].tolist())

num_samples = 1000
samples = [top_p_sample(logits, p=0.9, temperature=1.0) for _ in range(num_samples)]
samples_set = set(s if isinstance(s, int) else s.item() for s in samples)
outside_nucleus = samples_set - nucleus_tokens
print(f"Nucleus size: {nucleus_size} tokens out of {vocab_size}")
print(f"Unique tokens sampled: {len(samples_set)}")
assert len(outside_nucleus) == 0, \
    f"Sampled {len(outside_nucleus)} tokens outside nucleus: {list(outside_nucleus)[:5]}"
print(f"Test 2 PASSED: All {num_samples} samples came from within the nucleus")

# Test 3: Different p values should change nucleus size
token_low_p = top_p_sample(logits, p=0.1, temperature=1.0)
token_high_p = top_p_sample(logits, p=0.99, temperature=1.0)
print(f"\nTest 3: p=0.1 sampled token {token_low_p}, p=0.99 sampled token {token_high_p}")
print("Test 3 PASSED: Function works with different p values")

print("\nAll tests passed!")